# Task 1.2: Key Assumptions## Paper: Clustering Time Series Using Unsupervised-Shapelets**Authors**: Jesin Zakaria, Abdullah Mueen, Eamonn J. Keogh — ICDM 2012

## Assumption 1: Discriminative information resides in local subsequences**Assumption**: The method assumes that meaningful differences between clusters of time series can be captured by short local subsequences (shapelets) rather than the global shape or overall statistics of the time series.**Why the method needs it**: The entire u-shapelet discovery process — sliding a short window, computing sdist as a *minimum* over all alignments — is designed to detect local patterns. If the discriminative information were global (e.g., overall trend, frequency, or amplitude), the sdist operation would fail to capture it because it only looks at the best local match and ignores the rest of the time series.**Violation scenario**: Consider a dataset where two classes of time series both contain the same local motifs but differ only in their global statistics — for example, one class has a positive linear trend and the other has a negative linear trend, while both share identical local oscillation patterns. The u-shapelet method would find no discriminative local subsequence.**Paper reference**: Section I (Introduction) and Section III-A — the paper explicitly motivates u-shapelets by arguing that "a short pattern can capture the key distinctive features" of time series classes (paraphrased). The entire sdist definition (Definition 2) is built on this local-matching premise.

## Assumption 2: The gap metric correctly identifies meaningful cluster boundaries**Assumption**: The method assumes that the sdist values for a good u-shapelet will produce a bimodal distribution — one tight cluster of small distances (D_A) and one of large distances (D_B) — with a clear statistical gap between them.**Why the method needs it**: The gap metric `GAP = (mean(D_B) − std(D_B)) − (mean(D_A) + std(D_A))` is the *sole* criterion for evaluating candidate shapelets. If the true cluster structure does not produce bimodal sdist distributions (e.g., if distances form a unimodal or multi-modal distribution), the gap metric will either fail to find a valid split or produce a misleading one.**Violation scenario**: If the dataset contains three or more clusters that are equidistant in shapelet-distance space, the sorted sdist values would show a gradual increase rather than a sharp gap. The binary split assumption of the gap metric would then assign two clusters to the same group or split a single cluster.**Paper reference**: Definition 4 (Gap Score) in Section III-B; Figure 3 shows the idealised bimodal case. The paper acknowledges the binary split nature but addresses multi-class problems by iterative application.

## Assumption 3: Appropriate shapelet length range is known or can be estimated**Assumption**: The method assumes that a suitable range of shapelet lengths [l_min, l_max] is provided as input and that this range captures the scale of the discriminative patterns present in the data.**Why the method needs it**: The candidate generation step extracts all subsequences of lengths within [l_min, l_max]. If the true discriminative pattern is shorter than l_min, it will never be considered as a candidate. If it is longer than l_max, it will be broken into fragments that individually may not be discriminative. The algorithm has no adaptive mechanism to learn the optimal shapelet length from the data.**Violation scenario**: In a medical ECG dataset, the discriminative event might be a single sharp QRS complex lasting ~5 time points, but if the user sets l_min = 20, the algorithm will never consider candidates short enough to capture this pattern, leading to poor clustering quality.**Paper reference**: Section III-A describes the length parameters. The experimental setup in Section IV-A mentions setting l_min = n/20 and l_max = n/5 as defaults. This is a user-defined hyperparameter, and the paper does not propose an automatic selection method.

## Assumption 4: The greedy iterative peeling produces the correct cluster decomposition**Assumption**: The method assumes that greedily removing the most tightly clustered group at each step (based on the best u-shapelet) will correctly decompose the dataset into its natural clusters.**Why the method needs it**: The iterative peeling strategy is the mechanism by which the algorithm handles multi-class clustering. It assumes that the strongest separation at each step corresponds to a genuine cluster boundary. If a spurious pattern happens to produce a high gap score early on, the algorithm will incorrectly separate time series and all subsequent steps will operate on a corrupted remaining set.**Violation scenario**: Consider a dataset where two clusters are very similar to each other but both differ from a third. A spurious subsequence that appears by chance in a subset of examples from both similar clusters could produce a high gap score, merging members from different true clusters into a single group. This error cascades because the remaining set is now contaminated.**Paper reference**: Algorithm 1 in Section III-C describes the greedy loop. The paper does not provide theoretical guarantees on the correctness of greedy separation and relies on empirical validation.